<a href="https://colab.research.google.com/github/pallavinataraj8-glitch/Pallavi/blob/main/power_plant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install -q ultralytics paho-mqtt

In [8]:
import paho.mqtt.client as mqtt
BROKER = "broker.hivemq.com"
TOPIC  = "pallavi/plantwaterer"      # must be SAME in Wokwi
TARGET = "potted plant"              # what the camera looks for
ON_MSG = "water"                     # word sent to Wokwi

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.connect(BROKER, 1883, 60)
client.loop_start()

def send(msg):
    client.publish(TOPIC, msg)
    print("sent:", msg)

In [17]:
send("water")


sent: water


In [16]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
from ultralytics import YOLO
model = YOLO("yolov8n.pt")

def take_photo(filename="photo.jpg", quality=0.8):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const btn = document.createElement('button'); btn.textContent = 'Capture';
      div.appendChild(btn);
      const video = document.createElement('video'); video.style.display='block';
      const stream = await navigator.mediaDevices.getUserMedia({video:true});
      document.body.appendChild(div); div.appendChild(video);
      video.srcObject = stream; await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise(r => btn.onclick = r);
      const c = document.createElement('canvas');
      c.width = video.videoWidth; c.height = video.videoHeight;
      c.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop(); div.remove();
      return c.toDataURL('image/jpeg', quality);
    }''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    open(filename, 'wb').write(b64decode(data.split(',')[1]))
    return filename

photo = take_photo()
result = model(photo)[0]
names = [model.names[int(c)] for c in result.boxes.cls]
print("I see:", names)
if TARGET in names:
    send(ON_MSG)
else:
    print("No plant seen")

<IPython.core.display.Javascript object>


image 1/1 /content/photo.jpg: 480x640 1 potted plant, 1 vase, 1 teddy bear, 138.4ms
Speed: 4.6ms preprocess, 138.4ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)
I see: ['potted plant', 'teddy bear', 'vase']
sent: water
